In [5]:
import pandas as pd
import numpy as np

In [6]:
path = "/Users/coffeer/Documents/GitHub/R_lab1/Blockhouse-Work-Trial/data/AAPL_Quotes_Data.csv"
data = pd.read_csv(path)
# Suppose bid_price1 is the close price 
df = data[['timestamp', 'bid_price_1']].rename(columns = {'bid_price_1': 'close_price'})
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)
df

,close_price
timestamp,
2024-07-01 13:30:00+00:00,212.51
2024-07-01 13:31:00+00:00,213.07
2024-07-01 13:32:00+00:00,213.05
2024-07-01 13:33:00+00:00,213.25
2024-07-01 13:34:00+00:00,213.47
...,...
2024-08-30 19:55:00+00:00,229.20
2024-08-30 19:56:00+00:00,229.00
2024-08-30 19:57:00+00:00,229.00


## Snippet 3.1: Daily Volatility Estimates

In [7]:
def getDailyVol(close, span0=100):
    '''
    Calculate daily volatility estimates using exponentially weighted moving standard deviation.
    
    Input:
    - close: Pandas Series of closing prices indexed by date/time.
    - span0: Integer, span (number of days) for the EWM standard deviation (default=100).
    
    Output:
    - A Pandas Series containing daily volatility values indexed by date/time.
    '''

    # Step 1: Find the indices of the previous day's timestamps
    prev_day_indices = close.index.searchsorted(close.index - pd.Timedelta(days=1))
    prev_day_indices = prev_day_indices[prev_day_indices > 0]

    # Step 2: Map the indices to timestamps of the previous day
    prev_day_timestamps = pd.Series(
        close.index[prev_day_indices - 1], 
        index=close.index[close.shape[0] - prev_day_indices.shape[0]:]
    )

    # Step 3: Filter and align timestamps properly
    common_timestamps = prev_day_timestamps[prev_day_timestamps.isin(close.index)]
    aligned_current = common_timestamps.index.intersection(close.index)
    aligned_previous = common_timestamps.loc[aligned_current]

    # Ensure the alignment is correct
    daily_returns = pd.Series(close.loc[aligned_current].values / close.loc[aligned_previous].values - 1)

    # Step 4: Calculate exponentially weighted standard deviation of daily returns
    daily_volatility = daily_returns.ewm(span=100).std()

    return daily_volatility

### Test 

In [11]:
close = df['close_price']
(close[1] - close[0])/close[0]

/var/folders/kv/x8qrhxbj6gxf4kw707c07yvc0000gn/T/ipykernel_7742/3353373342.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  (close[1] - close[0])/close[0]


0.00263517010964191

In [8]:
getDailyVol(df['close_price'])

0             NaN
1        0.002788
2        0.002040
3        0.002151
4        0.002435
           ...   
16730    0.004292
16731    0.004282
16732    0.004295
16733    0.004324
16734    0.004346
Length: 16735, dtype: float64

## Snippet 3.2: Triple-Barrier Labeling Method

In [27]:
def applyPtSlOnT1(close, events, ptSl, molecule):
    """
    Apply the triple-barrier method to determine the first barrier touched.
    
    Input:
    - close: Pandas Series of closing prices.
    - events: DataFrame with 't1' (vertical barrier) and 'trgt' (target returns).
    - ptSl: List [upper_barrier_multiplier, lower_barrier_multiplier].
    - molecule: Subset of event indices to process.
    
    Output:
    - DataFrame with timestamps of barrier touches (if any).
    """
    events_ = events.loc[molecule]  # Subset of events
    out = events_[['t1']].copy()  # Initialize output DataFrame

    for loc, t1 in events_['t1'].items():  # Fixed here
        # Extract the price path from the event start to the vertical barrier
        path_prices = close.loc[loc:t1]
        
        # Calculate the barriers
        target = events_.loc[loc, 'trgt']
        upper_barrier = close[loc] * (1 + target * ptSl[0])
        lower_barrier = close[loc] * (1 - target * ptSl[1])
        
        # Check for barrier touches
        out.loc[loc, 'pt'] = path_prices[path_prices >= upper_barrier].index.min() if (path_prices >= upper_barrier).any() else np.nan
        out.loc[loc, 'sl'] = path_prices[path_prices <= lower_barrier].index.min() if (path_prices <= lower_barrier).any() else np.nan

    return out




### Test (this result does not look reasonable, will fix in tomorrow)

In [ ]:
close = pd.Series(
    [100, 105, 102, 108, 110],
    index=pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05'])
)


events = pd.DataFrame({
    't1': pd.to_datetime(['2023-01-04']),  # Vertical barrier
    'trgt': [0.05]  # 目标阈值
}, index=pd.to_datetime(['2023-01-01']))


ptSl = [1, 1]  
molecule = [pd.Timestamp('2023-01-01')] 

applyPtSlOnT1(close, events, ptSl, molecule)

,t1,pt,sl
2023-01-01,2023-01-04,2023-01-02,NaN


## Snippet 3.3: Getting the Time of First Touch

In [ ]:
def getEvents(close, tEvents, ptSl, trgt, minRet, numThreads, t1=False):
    """
    Label events using the triple-barrier method.

    Parameters:
    - close: pandas Series of prices
    - tEvents: pandas index of events
    - ptSl: list with two non-negative floats [pt, sl]
    - trgt: pandas Series of target returns
    - minRet: minimum return threshold
    - numThreads: number of threads for parallel processing
    - t1: pandas Series of vertical barriers (if False, vertical barriers are disabled)

    Returns:
    - events: DataFrame with 't1' (time of first touch) and 'trgt' (target return)
    """
    # Filter target returns that meet the minimum threshold
    trgt = trgt.loc[tEvents]
    trgt = trgt[trgt > minRet]  # Remove events with insufficient target returns

    # Define vertical barriers
    if t1 is False:
        t1 = pd.Series(pd.NaT, index=tEvents)  # No vertical barriers if t1 is False

    # Construct the events DataFrame
    side_ = pd.Series(1., index=trgt.index)  # Default side is neutral (1)
    events = pd.concat({'t1': t1, 'trgt': trgt, 'side': side_}, axis=1).dropna(subset=['trgt'])

    # Apply the triple-barrier method to calculate time of first touch
    df0 = applyPtSlOnT1(
        close=close,
        events=events,
        ptSl=ptSl,
        molecule=events.index
    )

    # Update events DataFrame with first touch times
    events['t1'] = df0.apply(lambda row: row.dropna().min(), axis=1)  # Get the earliest non-NaN time
    events = events.drop('side', axis=1)  # Remove the 'side' column, as it's not needed

    return events


### Test 

In [62]:
# Price series
close = pd.Series(
    [100, 105, 102, 108, 110],
    index=pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05'])
)

# Events timestamps
tEvents = pd.to_datetime(['2023-01-01', '2023-01-02'])

# Target returns
trgt = pd.Series(
    [0.1, 0.02],
    index=pd.to_datetime(['2023-01-01', '2023-01-02'])
)

# Stop-loss and profit-taking factors
ptSl = [1, 1]  # Enable both barriers

# Minimum return threshold
minRet = 0.01

# Number of threads (not used in this single-threaded example)
numThreads = 1

# Run the getEvents function with the test case
events = getEvents(close, tEvents, ptSl, trgt, minRet, numThreads)


print(events)


                   t1  trgt
2023-01-01 2023-01-05  0.10
2023-01-02 2023-01-03  0.02


##  Snippet 3.4: Adding a Vertical Barrier

In [63]:
def addVerticalBarrier(close, tEvents, numDays):
    """
    Add a vertical barrier to each event.

    Parameters:
    - close: pandas Series of prices (index is datetime).
    - tEvents: pandas DatetimeIndex of timestamps for the events.
    - numDays: int, number of days after which the vertical barrier is set.

    Returns:
    - t1: pandas Series of vertical barrier timestamps for each event.
    """
    # Find the index of the next price bar at or after numDays
    t1 = close.index.searchsorted(tEvents + pd.Timedelta(days=numDays))
    t1 = t1[t1 < close.shape[0]]  # Filter out indices that exceed the series range
    t1 = pd.Series(close.index[t1], index=tEvents[:t1.shape[0]])  # Map back to event timestamps
    return t1


### Test 

In [65]:
# Price series
close = pd.Series(
    [100, 105, 102, 108, 110],
    index=pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05'])
)

# Events timestamps
tEvents = pd.to_datetime(['2023-01-01', '2023-01-02'])

# Target returns
trgt = pd.Series(
    [0.1, 0.02],
    index=pd.to_datetime(['2023-01-01', '2023-01-02'])
)

# Stop-loss and profit-taking factors
ptSl = [1, 1]  # Enable both barriers

# Minimum return threshold
minRet = 0.01

# Number of threads (not used in this single-threaded example)
numThreads = 1

# Number of days for the vertical barrier
numDays = 2

# Add vertical barrier
t1 = addVerticalBarrier(close, tEvents, numDays)

# Run the getEvents function with the vertical barrier
events = getEvents(close, tEvents, ptSl, trgt, minRet, numThreads, t1)

# Print the resulting events DataFrame
print(events)


                   t1  trgt
2023-01-01 2023-01-03  0.10
2023-01-02 2023-01-03  0.02


## Snippet 3.5: Labeling for Side and Size

In [76]:
def getBins(events, close):
    """
    Label events for side and size using the triple-barrier method.

    Parameters:
    - events: DataFrame with columns ['t1'] (time of first touch) and ['trgt'].
    - close: pandas Series of prices (index is datetime).

    Returns:
    - DataFrame with columns:
        - 'ret': The return realized at the time of the first touched barrier.
        - 'bin': The label {1, -1, 0} as a function of the sign of the outcome.
    """
    # Step 1: Align prices with events
    events_ = events.dropna(subset=['t1'])  # Drop events where t1 is NaN
    px = events_.index.union(events_['t1'].values).drop_duplicates()  # Union of start and end times
    px = close.reindex(px, method='bfill')  # Align prices with these timestamps

    # Step 2: Calculate returns and assign labels
    out = pd.DataFrame(index=events_.index)
    out['ret'] = px.loc[events_['t1'].values].values / px.loc[events_.index].values - 1  # Returns
    out['bin'] = np.sign(out['ret'])  # Labels based on return sign
    return out

### Test 

In [78]:
# Price series
close = pd.Series(
    [100, 105, 102, 108, 110],
    index=pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05'])
)

# Events DataFrame
events = pd.DataFrame({
    't1': pd.to_datetime(['2023-01-03', '2023-01-04']),  # End time of events
    'trgt': [0.1, 1]  # Target returns
}, index=pd.to_datetime(['2023-01-01', '2023-01-02']))  # Start time of events

# Run the getBins function
bins = getBins(events, close)

# Print the resulting labels
print(bins)



                 ret  bin
2023-01-01  0.020000  1.0
2023-01-02  0.028571  1.0


## Snippet 3.6: Expanding getEvents for Meta-Labeling

In [79]:
def getEvents(close, tEvents, ptSl, trgt, minRet, numThreads, t1=False, side=None):
    """
    Generate events with optional meta-labeling support.

    Parameters:
    - close: pandas Series of prices (index is datetime).
    - tEvents: pandas index of event timestamps.
    - ptSl: list of floats [pt, sl] for profit-taking and stop-loss multipliers.
    - trgt: pandas Series of target returns.
    - minRet: minimum return threshold.
    - numThreads: number of threads for parallel processing.
    - t1: pandas Series of vertical barrier timestamps.
    - side: pandas Series of trade sides (1 for long, -1 for short, None for neutral).

    Returns:
    - events: DataFrame with columns ['t1', 'trgt', 'side'].
    """
    # Step 1: Filter events by target return
    trgt = trgt.loc[tEvents]
    trgt = trgt[trgt > minRet]  # Filter out events with insufficient target returns

    # Step 2: Define vertical barriers
    if t1 is False:
        t1 = pd.Series(pd.NaT, index=tEvents)  # No vertical barriers if t1 is False

    # Step 3: Adjust horizontal barriers and create the events DataFrame
    if side is None:
        _ptSl = pd.Series([ptSl[0], ptSl[0]], index=['pt', 'sl'])  # Symmetric barriers
        side_ = pd.Series(1.0, index=trgt.index)  # Default side is neutral
    else:
        _ptSl = side.loc[trgt.index] * pd.Series(ptSl[:2], index=['pt', 'sl'])  # Adjust barriers by side
        side_ = side.loc[trgt.index]  # Use the provided trade sides

    events = pd.concat({'t1': t1, 'trgt': trgt, 'side': side_}, axis=1).dropna(subset=['trgt'])

    # Step 4: Apply horizontal and vertical barriers
    df0 = applyPtSlOnT1(
        close=close,
        events=events,
        ptSl=[_ptSl['pt'], _ptSl['sl']],
        molecule=events.index
    )

    # Update t1 with the earliest barrier breach
    events['t1'] = df0.dropna(how='all').min(axis=1)

    # Drop 'side' column if no meta-labeling
    if side is None:
        events = events.drop('side', axis=1)

    return events


### Test 

In [81]:
# Price series
close = pd.Series(
    [100, 105, 102, 108, 110],
    index=pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05'])
)

# Event timestamps
tEvents = pd.to_datetime(['2023-01-01', '2023-01-02'])

# Target returns
trgt = pd.Series(
    [0.1, 0.5],
    index=pd.to_datetime(['2023-01-01', '2023-01-02'])
)

# Trade sides (1 for long, -1 for short)
side = pd.Series(
    [1, -1],
    index=pd.to_datetime(['2023-01-01', '2023-01-02'])
)

# Stop-loss and profit-taking factors
ptSl = [1, 1]  # Enable both barriers

# Minimum return threshold
minRet = 0.04

# Number of threads
numThreads = 1

# Add vertical barrier
t1 = addVerticalBarrier(close, tEvents, numDays=2)

# Run the extended getEvents function
events = getEvents(close, tEvents, ptSl, trgt, minRet, numThreads, t1, side)

# Print the resulting events DataFrame
print(events)



                   t1  trgt  side
2023-01-01 2023-01-03   0.1     1
2023-01-02 2023-01-04   0.5    -1


## Snippet 3.7: Expanding getBins for Meta-Labeling

In [73]:
def getBins(events, close):
    """
    Compute event's outcome (including side information, if provided).

    Parameters:
    - events: DataFrame with columns ['t1', 'trgt', 'side'].
    - close: pandas Series of prices (index is datetime).

    Returns:
    - DataFrame with columns:
        - 'ret': The return realized at the time of the first touched barrier.
        - 'bin': The label {1, 0}.
            - 1: Profit aligned with side (meta-labeling).
            - 0: Loss or vertical barrier reached.
    """
    # Step 1: Align prices with events
    events_ = events.dropna(subset=['t1'])  # Drop events where t1 is NaN
    px = events_.index.union(events_['t1'].values).drop_duplicates()  # Union of start and end times
    px = close.reindex(px, method='bfill')  # Align prices with these timestamps

    # Step 2: Create output DataFrame
    out = pd.DataFrame(index=events_.index)
    out['ret'] = px.loc[events_['t1'].values].values / px.loc[events_.index].values - 1  # Returns

    # Step 3: Adjust returns and labels based on 'side'
    if 'side' in events_:
        out['ret'] *= events_['side']  # Adjust return by side (direction)
        out['bin'] = np.where(out['ret'] > 0, 1, 0)  # Labels: 1 for positive, 0 for negative or vertical barrier
    else:
        out['bin'] = np.sign(out['ret'])  # Default: Labels {1, -1} based on return sign

    return out


### Test 

In [74]:
# New Price series with neutral outcome
close = pd.Series(
    [100, 105, 102, 105, 105],
    index=pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04', '2023-01-05'])
)

# Events DataFrame remains the same
events = pd.DataFrame({
    't1': pd.to_datetime(['2023-01-03', '2023-01-04']),  # End time of events
    'trgt': [0.1, 0.02],  # Target returns
    'side': [1, -1]  # Trade sides: 1 for long, -1 for short
}, index=pd.to_datetime(['2023-01-01', '2023-01-02']))  # Start time of events

# Run the expanded getBins function
bins = getBins(events, close)

# Print the resulting labels
print(bins)


             ret  bin
2023-01-01  0.02    1
2023-01-02 -0.00    0
